# Monte Carlo Risk Assessment for Portfolio Management

## Table of Contents

1. Setting up the environment
2. Project configuration
3. Download and prepare the stock data
4. Build the Monte Carlo return simulations
5. Calculate VaR and CVaR
6. Test VaR forecasts against actual returns
7. Run the workflow end-to-end
8. Conclusions

## Project Objective

FFF

## 1. Setting Up the Environment

In [ ]:
import datetime as dt
import numpy as np
import pandas as pd
import yfinance as yf

from data import build_window_dates, get_data, get_historical_portfolio_returns
from simulation import (
    build_mean_matrix,
    simulate_normal_model,
    simulate_random_day_model,
    simulate_student_t_model,
)
from risk import summarize_risk
from backtesting import run_var_backtest

### Importing Libraries
- `datetime (dt)`: For handling date and time operations
- `numpy (np)`: For numerical arrays and linear algebra used in simulations and risk calculations
- `pandas (pd)`: For organizing stock data, returns, and tables in DataFrames and Series
- `yfinance (yf)`: For downloading historical stock-price data from Yahoo Finance

## 2. Project Configuration

These stocks were selected because _________

The weights are evenly distributed as a control group, and the effects of varying distributions of each stock's weight in the portfolio can be determined through further analysis.  

In [ ]:
stocks = ['AAPL', 'MSFT', 'GOOG', 'AMZN']
weights = [0.25, 0.25, 0.25, 0.25]
initial_portfolio = 1000

#### Defining Parameters:

- `stocks`: Tickers for the stocks included in the portfolio
- `weights`: Portfolio weight assigned to each stock
- `initial_portfolio`: Starting amount of money in the portfolio

## 3. Download and Prepare the Stock Data

#### Defining the Analysis Date Window

To determine the exact start date (`start_date`) and end date (`end_date`), an appropriate sample size is required. The end date was hardcoded to September 30, 2026. A window of 300 days (`days_back=300`) encapsulates more than the past year of trading days.

- Set the fixed end date to September 30, 2026
- Calculate the start date by subtracting `days_back`

In [ ]:
def build_window_dates(days_back=300):
    end_date = dt.datetime(2026, 9, 30) #9/30/2026
    start_date = end_date - dt.timedelta(days=days_back)
    return start_date, end_date


start_date, end_date = build_window_dates(days_back=300)

#### Defining Parameters:

- `days_back`: Number of days before the fixed end date used to determine the start date

#### Defining the Stock-Data Function

Analyzing the returns of individual stocks requires the stock tickers (`stocks`), start date (`start`), end date (`end`), and trading-day count (`trade_days=200`).

- Download stock data from Yahoo Finance
- Select closing prices
- Raise an error if closing prices are unavailable
- Restrict the data to the final 200 trading days
- Compute percentage changes
- Calculate mean daily returns
- Calculate the covariance matrix

In [ ]:
def get_data(stocks, start, end, trade_days=200):
    stock_data = yf.download(stocks, start=start, end=end, auto_adjust=False) #download stock data from Yahoo Finance

    if isinstance(stock_data.columns, pd.MultiIndex): #if stock data has multiple colummns
        stock_data = stock_data['Close'] #select column labeled 'Close'
    else: #if stock data has 1 column
        if 'Close' in stock_data.columns: #if the 1 column is labeled 'Close'
            stock_data = stock_data[['Close']] #select the column labeled 'Close'
        else: #if the 1 column is not labeled 'Close'
            raise ValueError("The downloaded data does not contain a 'Close' column.") #raise an error
        
    #stock data is now only closing values for each day in 200-day span

    returns = stock_data.pct_change().tail(trade_days) #compute percentage change in closing value for each day in 200-day span
    mean_returns = returns.mean() #mean of daily returns for the stock
    cov_matrix = returns.cov() #covariance matrix of daily returns for the stock
    return mean_returns, cov_matrix, returns

#### Defining Parameters:

- `start`: Beginning date of the stock-data period
- `end`: Ending date of the stock-data period
- `trade_days`: Number of recent trading days included in the analysis

#### Downloading Historical Portfolio Returns

- Set the fixed end date
- Calculate the start date five years before the fixed end date
- Download closing prices for each stock
- Calculate percentage changes
- Multiply daily stock returns by portfolio weights

In [ ]:
def get_historical_portfolio_returns(stocks, weights, years_back=5):
    end_date = pd.Timestamp("2026-09-30").normalize() #fixed end date: 9/30/2026
    start_date = end_date - pd.DateOffset(years=years_back) #date 5 years before the fixed end date

    prices = yf.download(stocks,start=start_date,end=end_date,auto_adjust=True,)['Close'] #closing prices for each stock on each day in 5-year span

    returns = prices.pct_change().dropna() #percentage change in closing value for each stock and for each day in 5-year span
    portfolio_returns = returns @ weights #multiplies the returns of each stock by its weight , sums returns for each day to get total portfolio return for each day in 5-year span
    return returns, portfolio_returns

#### Defining Parameters:

- `years_back`: Number of years of historical data to download

## 4. Build the Monte Carlo Return Simulations

Creates a column vector with each stock's mean daily return and repeats the vector across `num_days` columns.

- Convert mean daily returns into a NumPy array
- Repeat the mean-return column across `num_days` columns

In [ ]:
num_days = 50


def build_mean_matrix(mean_returns, num_days):
    mean_values = np.asarray(mean_returns) #converting mean_returns to a column vector of shape (n_stocks, 1)
    mean_returns_matrix = np.tile(mean_values.reshape(-1, 1), (1, num_days)) #repeating mean_values column vector num_days times so each column is identical
    return mean_returns_matrix

#### Defining Parameters:

- `num_days`: Number of trading days in the simulation horizon

### Run Monte Carlo Simulation Using a Multivariate Normal Distribution

- Create a zero-filled portfolio simulation matrix
- Repeat the simulation for each Monte Carlo path
- Generate standard normal variables
- Apply covariance through Cholesky decomposition
- Calculate daily stock and portfolio returns
- Compound portfolio value across the horizon

In [ ]:
mc_sims = 500


def simulate_normal_model(mean_m, cov_matrix, weights, initial_portfolio, num_days, mc_sims):
    portfolio_sims = np.full(shape=(num_days, mc_sims), fill_value=0.0) #creates all 0.0 matrix, rows = num_days, columns = mc_sims

    for m in range(mc_sims): #for each column/simulation
        z = np.random.normal(size=(num_days, len(weights))) #array of standard normal vars dimensions (num_days, n_stocks)
        l = np.linalg.cholesky(cov_matrix) #Cholesky decomposition of the covariance matrix comparing stocks to each other
        daily_returns = mean_m + l @ z.T #cholesky matrix*standard normal array + mean returns matrix = daily returns for each stock for each day
        portfolio_sims[:, m] = np.cumprod(daily_returns.T @ weights + 1) * initial_portfolio #for each day, multiply each stock's return by its weight to get the days full portfolio return, repeat for each day

    return portfolio_sims

#### Defining Parameters:

- `mean_m`: Matrix of mean stock returns repeated across the simulation period
- `cov_matrix`: Covariance matrix of stock returns
- `mc_sims`: Number of Monte Carlo simulation paths

### Run Monte Carlo Simulation Using a Multivariate Student-t Distribution

- Create a random-number generator if needed
- Generate standard normal variables
- Generate chi-square variables
- Apply scaled Cholesky decomposition
- Calculate daily stock and portfolio returns
- Compound portfolio value across the horizon

In [ ]:
def simulate_student_t_model(mean_m, cov_matrix, weights, initial_portfolio, num_days, mc_sims, df=8, rng=None):
    if rng is None: #if no random number generator is provided
        rng = np.random.default_rng() #create default random number generator

    portfolio_sims_t = np.full(shape=(num_days, mc_sims), fill_value=0.0)

    for m in range(mc_sims):
        z = np.random.normal(size=(num_days, len(weights)))
        u = rng.chisquare(df, size=num_days).reshape(num_days, 1) #generate chi-square random variables
        l = np.linalg.cholesky(cov_matrix * (df - 2) / df) #cholesky decomposition scaled by (df-2)/df to account for the Student-t distribution's heavier tails
        x = z / np.sqrt(u / df) #z-term adjusted so the covariance is the same as the multivariate normal distribution
        daily_returns = mean_m + l @ x.T
        portfolio_sims_t[:, m] = np.cumprod(daily_returns.T @ weights + 1) * initial_portfolio

    return portfolio_sims_t

#### Defining Parameters:

- `df`: Degrees of freedom controlling the thickness of the return-distribution tails
- `rng`: Optional random-number generator used for reproducible simulations

### Sample Historical Return Days with Replacement to Build a Distribution

- Randomly sample historical return days
- Compound sampled returns using portfolio weights

In [ ]:
def simulate_random_day_model(returns, weights, initial_portfolio, num_days, mc_sims):
    portfolio_hist = np.full(shape=(num_days, mc_sims), fill_value=0.0)

    for m in range(mc_sims):
        sampled_returns = returns.sample(n=num_days, replace=True) #randomly select certain days from historical returns and use their real returns as the returns for each stock
        portfolio_hist[:, m] = np.cumprod(sampled_returns @ weights + 1) * initial_portfolio #randomly selected returns for each stock put together to completely simulate stock returns for each day

    return portfolio_hist

## 5. Calculate VaR and CVaR

### Value at Risk: Return the Percentile for a Given Confidence Level

- Return the alpha percentile for a Pandas Series
- Raise an error for unsupported input types

In [ ]:
alpha = 5


def mcValueRisk(returns, alpha=5):
    if isinstance(returns, pd.Series): 
        return np.percentile(returns, alpha) #return the alpha percentile of the returns
    raise TypeError("Expected pandas data series") #raise an error if the input is not a pandas series

#### Defining Parameters:

- `alpha`: Tail percentile used to calculate VaR

### Compute CVaR / Expected Shortfall Below the VaR Threshold

- Identify returns below the VaR threshold
- Return the mean of those returns
- Raise an error for unsupported input types

In [ ]:
def mcConditionalValueRisk(returns, alpha=5):
    if isinstance(returns, pd.Series):
        below_var = returns <= mcValueRisk(returns, alpha=alpha) #generate all returns below alpha percentile of returns
        return returns[below_var].mean() #return mean of all returns below alpha percentile of returns
    raise TypeError("Expected pandas data series")

### Summarize VaR and CVaR for Each Model Column

- Calculate VaR for each model
- Calculate CVaR for each model
- Return a DataFrame containing both measures

In [ ]:
def summarize_risk(results_df, initial_portfolio, alpha=5):
    VaR = results_df.apply(lambda col: initial_portfolio - mcValueRisk(col, alpha=alpha)) #for each distribution type, determine VaR by subtracting alpha percentile final return from initial portfolio value
    CondVaR = results_df.apply(lambda col: initial_portfolio - mcConditionalValueRisk(col, alpha=alpha)) #for each distribution type, determine CVaR by subtracting mean of returns below alpha percentile from initial portfolio value
    return pd.DataFrame({'VaR': VaR, 'CVaR': CondVaR})

## 6. Test VaR Forecasts Against Actual Returns

### Backtest VaR Models Against Realized Portfolio Returns

- Initialize the backtest table and random-number generator
- Store actual portfolio returns
- Iterate across each backtesting date
- Select each historical window
- Apply normal, Student-t, and historical VaR forecasts
- Mark violations and calculate violation rates

In [ ]:
def run_var_backtest(stock_returns, portfolio_returns, weights, window_backtest=250, alpha_backtest=0.05, num_sims_backtest=10000, df_backtest=8, rng=None):
    if rng is None:  # if no rng is provided
        rng = np.random.default_rng(10)  # create new rng

    backtest_data = pd.DataFrame(
        index=stock_returns.index[window_backtest:],  # index by date, only uses dates after the initial window_backtest period
        columns=['Normal', 'Student-t', 'Historical'],
        dtype=float,
    )
    backtest_data['Actual Return'] = portfolio_returns.iloc[window_backtest:].to_numpy()  # adds actual portfolio returns for each day in the backtest period

    stocks = stock_returns.columns  # stocks are the same as the column names in the stock-returns DataFrame

    for i in range(window_backtest, len(stock_returns)):  # for each day in the backtest period
        historical_window = stock_returns.iloc[i - window_backtest:i]  # historical window of stock returns is 250 days before the current day being evaluated
        mu = historical_window.mean().to_numpy()  # average return for each stock over the historical window
        cov = historical_window.cov().to_numpy()  # covariance matrix of returns for each stock over the historical window
        historical_portfolio = historical_window @ weights  # full portfolio returns for each day
        L = np.linalg.cholesky(cov)  # Cholesky decomposition of the covariance matrix

        z_normal = rng.standard_normal(size=(num_sims_backtest, len(stocks)))  # standard normal random variables for each stock for each simulation
        simulated_normal = mu + z_normal @ L.T  # simulate daily returns for each stock
        portfolio_normal = simulated_normal @ weights  # generates full portfolio returns for each day
        backtest_data.iloc[i - window_backtest, 0] = np.quantile(portfolio_normal, alpha_backtest)  # calculates VaR (bottom alpha returns) out of the simulated portfolio end returns

        L_t = np.linalg.cholesky(cov * (df_backtest - 2) / df_backtest)  # cholesky decompsitions scaled by (df-2)/df to account for the Student-t distribution's heavier tails
        z_t = rng.standard_normal(size=(num_sims_backtest, len(stocks))) @ L_t.T  # standard normal random vars multiplied by cholesky decomposition
        u = rng.chisquare(df_backtest, size=num_sims_backtest)  # chi-square random variables for each simulation
        simulated_t = mu + z_t / np.sqrt(u / df_backtest)[:, None]  # simulate daily returns for each stock under the Student-t distribution
        portfolio_t = simulated_t @ weights  # full-portfolio returns for each day
        backtest_data.iloc[i - window_backtest, 1] = np.quantile(portfolio_t, alpha_backtest)  # VaR

        backtest_data.iloc[i - window_backtest, 2] = np.quantile(historical_portfolio, alpha_backtest)  # VaR of actual data for each day in the backtest period

    for model in ['Normal', 'Student-t', 'Historical']:  # for each model type
        backtest_data[f'{model} Violation (T/F)'] = (backtest_data['Actual Return'] < backtest_data[model])  # if the actual return is less than the VaR for that day, it is a violation

    violation_rates = backtest_data.iloc[:, 4:7].mean().mul(100)
    summary = pd.DataFrame({
        'Model': ['Normal', 'Student-t', 'Historical'],
        'Expected Violations (%)': [5.0, 5.0, 5.0],
        'Actual Violations (%)': violation_rates.values,  # percentage of True's in each column (% of days actual returns < VaR)
    })

    return backtest_data, summary

#### Defining Backtesting Parameters:

- `window_backtest`: Number of historical observations in each rolling window
- `alpha_backtest`: Tail probability used to calculate backtested VaR
- `num_sims_backtest`: Number of simulations used for each VaR forecast
- `df_backtest`: Degrees of freedom for the Student-t backtesting model
- `rng`: Optional random-number generator used for reproducible backtesting

## 7. Run the Workflow End-to-End

- Build the date window and mean-return matrix
- Run all Monte Carlo simulation models
- Create final portfolio-value results
- Calculate VaR and CVaR
- Download historical returns
- Run and print the backtest summary

In [ ]:
def main():
    start_date, end_date = build_window_dates(days_back=300)
    mean_returns, cov_matrix, returns = get_data(stocks, start_date, end_date) #returning values for chosen stocks over the last 300 days
    mean_m = build_mean_matrix(mean_returns, num_days) #mean_returns column vector repeated num_days times

    normal_sims = simulate_normal_model(mean_m, cov_matrix, weights, initial_portfolio, num_days, mc_sims) #simulates full portfolio returns for each day
    t_sims = simulate_student_t_model(mean_m, cov_matrix, weights, initial_portfolio, num_days, mc_sims, df=8)
    hist_sims = simulate_random_day_model(returns, weights, initial_portfolio, num_days, mc_sims)

    results_df = pd.DataFrame({ #creates dataframe with the final portfolio value for each simulation for each model
        'normal': normal_sims[-1, :],
        't': t_sims[-1, :],
        'hist': hist_sims[-1, :],
    })

    risk_summary = summarize_risk(results_df, initial_portfolio, alpha=alpha) #generate VaR and CVaR for each model based on final portfolio returns
    print('Risk Summary')
    print(risk_summary)

    returns_past_data, portfolio_returns_past_data = get_historical_portfolio_returns(stocks, weights, years_back=5)
    backtest_data, summary = run_var_backtest(
        returns_past_data,
        portfolio_returns_past_data,
        weights,
        window_backtest=250, #number of days to use for historical data to calculate VaR
        alpha_backtest=0.05,
        num_sims_backtest=10000,
        df_backtest=8, #same df as used in the Student-t model
    )

    print('\nVaR Backtest Summary')
    print(summary)

### Coordinate the data, simulation, risk, and backtesting steps using the project configuration.

In [ ]:
if __name__ == '__main__': #if the file is being run directly
    main()

## 8. Conclusions

Descriptions of findings here